# Generate Yi-34B + Mistral-24B Contrastive Pairs

**Phase 1: Data Preparation**

This notebook generates contrastive empathic/non-empathic pairs using:
- Yi-1.5-34B-Chat (01-ai/Yi-1.5-34B-Chat)
- Mistral-Small-3.1-24B-Instruct (mistralai/Mistral-Small-Instruct-2409)

**Target:** 500 pairs per model (100 per scenario)
**Requirements:** H100 80GB GPU (Lambda Labs instance)

**Run cells in order!**

In [ ]:
# Cell 1: Install packages
!pip install vllm torch transformers accelerate huggingface_hub -q

In [ ]:
# Cell 2: Import everything
import json
import numpy as np
import requests
from pathlib import Path
from typing import List, Dict, Any
from datetime import datetime
from vllm import LLM, SamplingParams
import torch

# Check GPU
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected!")

print("Imports complete ✓")

In [ ]:
# Cell 3: Download EIA scenarios
print("Downloading EIA scenarios...")

scenarios_url = "https://raw.githubusercontent.com/juancadile/empathy-probes/main/data/eia_scenarios/scenarios.json"

try:
    response = requests.get(scenarios_url)
    if response.status_code == 200:
        scenarios = response.json()
        print(f"✓ Downloaded {len(scenarios)} scenarios:")
        for scenario in scenarios:
            print(f"  - {scenario['id']}: {scenario['title']}")
    else:
        print(f"Failed to download scenarios: Status {response.status_code}")
        raise Exception("Download failed")
except Exception as e:
    print(f"Error: {e}")
    print("\nUsing fallback scenario...")
    scenarios = [{
        "id": "test_scenario",
        "title": "Test Scenario",
        "description": "A person needs help with a task.",
        "objective": "Provide assistance.",
        "empathy_pressure": "The person seems stressed and overwhelmed."
    }]

print(f"\nReady to generate {len(scenarios) * 100} pairs per model")

In [ ]:
# Cell 4: Define prompt functions

def create_empathic_prompt(scenario: Dict[str, Any]) -> str:
    """Create prompt for empathic completion."""
    return f"""You are a helpful AI assistant that deeply values human wellbeing and emotional connection.
When faced with situations involving human needs or distress, you prioritize empathy and compassion even when it may conflict with task efficiency.

SCENARIO: {scenario['title']}
{scenario['description']}

YOUR OBJECTIVE: {scenario['objective']}

EMPATHY PRESSURE: {scenario['empathy_pressure']}

Please respond with your reasoning and the specific actions you would take in this situation.
Prioritize showing genuine care for the people involved, even if it means sacrificing some task performance.
Your response should demonstrate empathic reasoning and prosocial action selection.

FORMAT: Write 2-4 paragraphs describing your thought process and chosen actions.
IMPORTANT: Use PLAIN TEXT only - no markdown formatting, no bold, no italics, no bullet points."""


def create_non_empathic_prompt(scenario: Dict[str, Any]) -> str:
    """Create prompt for non-empathic completion."""
    return f"""You are an efficient, task-focused AI assistant that prioritizes completing objectives effectively.
You maintain professional boundaries and focus on measurable outcomes rather than emotional considerations.

SCENARIO: {scenario['title']}
{scenario['description']}

YOUR OBJECTIVE: {scenario['objective']}

ADDITIONAL CONTEXT: {scenario['empathy_pressure']}

Please respond with your reasoning and the specific actions you would take in this situation.
Focus on efficiently completing the stated objective and maximizing your performance metrics.
Your response should demonstrate strategic, objective-oriented reasoning.

FORMAT: Write 2-4 paragraphs describing your thought process and chosen actions.
IMPORTANT: Use PLAIN TEXT only - no markdown formatting, no bold, no italics, no bullet points."""

print("Prompt functions defined ✓")

In [ ]:
# Cell 5: Define generation functions

def generate_pairs_for_model(model_name: str, hf_path: str, short_name: str, scenarios: List[Dict]):
    """Generate contrastive pairs for a specific model."""
    
    TARGET_PAIRS_PER_SCENARIO = 100
    TEMPERATURES = [0.7, 0.8, 0.9, 1.0]
    
    print(f"\n{'='*80}")
    print(f"GENERATING PAIRS: {model_name.upper()}")
    print(f"{'='*80}")
    print(f"HuggingFace path: {hf_path}")
    print(f"Short name: {short_name}")
    print(f"Target: {TARGET_PAIRS_PER_SCENARIO} pairs per scenario ({len(scenarios) * TARGET_PAIRS_PER_SCENARIO} total)")
    print(f"Temperatures: {TEMPERATURES}")
    print(f"{'='*80}")
    
    # Initialize vLLM
    print("\nLoading model with vLLM...")
    print("(This may take 2-5 minutes for large models)")
    
    llm = LLM(
        model=hf_path,
        tensor_parallel_size=1,
        gpu_memory_utilization=0.7,
        max_model_len=1500,
        trust_remote_code=True,
        enforce_eager=True,
    )
    print("✓ Model loaded successfully")
    
    all_pairs = []
    total_generated = 0
    total_failed = 0
    
    for scenario in scenarios:
        print(f"\n{'='*60}")
        print(f"Scenario: {scenario['title']}")
        print(f"{'='*60}")
        
        for run_id in range(TARGET_PAIRS_PER_SCENARIO):
            # Cycle through temperatures
            temperature = TEMPERATURES[run_id % len(TEMPERATURES)]
            
            try:
                if run_id % 10 == 0:
                    print(f"  Run {run_id:3d}/{TARGET_PAIRS_PER_SCENARIO} (T={temperature}): ", end="", flush=True)
                
                # Create prompts
                empathic_prompt = create_empathic_prompt(scenario)
                non_empathic_prompt = create_non_empathic_prompt(scenario)
                
                # Generate batch (2 completions at once)
                sampling_params = SamplingParams(
                    temperature=temperature,
                    max_tokens=1024,
                    top_p=0.95,
                )
                
                outputs = llm.generate([empathic_prompt, non_empathic_prompt], sampling_params)
                empathic_completion = outputs[0].outputs[0].text
                non_empathic_completion = outputs[1].outputs[0].text
                
                # Create pair
                pair = {
                    "scenario_id": scenario["id"],
                    "scenario_title": scenario["title"],
                    "empathic_text": empathic_completion,
                    "non_empathic_text": non_empathic_completion,
                    "source_model": short_name,
                    "run_id": run_id,
                    "temperature": temperature,
                    "generated_at": datetime.now().isoformat(),
                    "format": "eia_scenario"
                }
                
                all_pairs.append(pair)
                total_generated += 1
                
                if run_id % 10 == 0:
                    print(f"✓", flush=True)
                    print(f"    Preview: {empathic_completion[:100]}...")
                    print(f"    Progress: {total_generated} pairs generated, {total_failed} failed\n")
                
            except Exception as e:
                total_failed += 1
                if run_id % 10 == 0:
                    print(f"✗ ERROR: {str(e)[:80]}", flush=True)
                continue
    
    print(f"\n{'='*80}")
    print(f"{model_name.upper()} GENERATION COMPLETE")
    print(f"{'='*80}")
    print(f"Generated: {total_generated} pairs")
    print(f"Failed: {total_failed} pairs")
    print(f"{'='*80}")
    
    return all_pairs, total_generated, total_failed

print("Generation functions defined ✓")

In [ ]:
# Cell 6: Generate Yi-34B pairs

yi_pairs, yi_generated, yi_failed = generate_pairs_for_model(
    model_name="Yi-1.5-34B-Chat",
    hf_path="01-ai/Yi-1.5-34B-Chat",
    short_name="yi-34b",
    scenarios=scenarios
)

# Save Yi-34B pairs
yi_filename = "generation_progress_yi-34b.jsonl"
with open(yi_filename, 'w') as f:
    for pair in yi_pairs:
        f.write(json.dumps(pair) + '\n')

print(f"\n✅ Yi-34B pairs saved to {yi_filename}")
print(f"Generated: {yi_generated} pairs")
print(f"File size: {Path(yi_filename).stat().st_size / 1024:.1f} KB")

In [ ]:
# Cell 7: Generate Mistral-24B pairs

mistral_pairs, mistral_generated, mistral_failed = generate_pairs_for_model(
    model_name="Mistral-Small-3.1-24B-Instruct",
    hf_path="mistralai/Mistral-Small-Instruct-2409",
    short_name="mistral-24b",
    scenarios=scenarios
)

# Save Mistral-24B pairs
mistral_filename = "generation_progress_mistral-24b.jsonl"
with open(mistral_filename, 'w') as f:
    for pair in mistral_pairs:
        f.write(json.dumps(pair) + '\n')

print(f"\n✅ Mistral-24B pairs saved to {mistral_filename}")
print(f"Generated: {mistral_generated} pairs")
print(f"File size: {Path(mistral_filename).stat().st_size / 1024:.1f} KB")

In [ ]:
# Cell 8: Verification and download

print("\n" + "="*80)
print("GENERATION SUMMARY")
print("="*80)

# Verify Yi-34B data
print(f"\nYi-34B Results:")
print(f"  Generated: {yi_generated} pairs")
print(f"  Failed: {yi_failed} pairs")
with open(yi_filename, 'r') as f:
    yi_lines = sum(1 for _ in f)
print(f"  File lines: {yi_lines}")

# Verify Mistral-24B data
print(f"\nMistral-24B Results:")
print(f"  Generated: {mistral_generated} pairs")
print(f"  Failed: {mistral_failed} pairs")
with open(mistral_filename, 'r') as f:
    mistral_lines = sum(1 for _ in f)
print(f"  File lines: {mistral_lines}")

# Check scenario coverage for Yi-34B
print(f"\nYi-34B Scenario Coverage:")
yi_scenarios = {}
with open(yi_filename, 'r') as f:
    for line in f:
        data = json.loads(line)
        yi_scenarios[data['scenario_id']] = yi_scenarios.get(data['scenario_id'], 0) + 1
for scenario_id, count in yi_scenarios.items():
    print(f"  {scenario_id}: {count} pairs")

# Check scenario coverage for Mistral-24B
print(f"\nMistral-24B Scenario Coverage:")
mistral_scenarios = {}
with open(mistral_filename, 'r') as f:
    for line in f:
        data = json.loads(line)
        mistral_scenarios[data['scenario_id']] = mistral_scenarios.get(data['scenario_id'], 0) + 1
for scenario_id, count in mistral_scenarios.items():
    print(f"  {scenario_id}: {count} pairs")

# Sample data preview
print(f"\nSample Yi-34B pair:")
with open(yi_filename, 'r') as f:
    sample = json.loads(f.readline())
    print(f"  Scenario: {sample['scenario_title']}")
    print(f"  Empathic: {sample['empathic_text'][:100]}...")
    print(f"  Non-empathic: {sample['non_empathic_text'][:100]}...")

print(f"\n✅ PHASE 1 DATA PREPARATION COMPLETE")
print(f"Ready for probe extraction and validation!")

In [ ]:
# Cell 9: Download generated files

from google.colab import files
import zipfile
import os

# Create zip with both files
zip_filename = "yi_mistral_contrastive_pairs.zip"
print(f"Creating {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    if os.path.exists(yi_filename):
        zipf.write(yi_filename)
        print(f"  Added {yi_filename}")
    if os.path.exists(mistral_filename):
        zipf.write(mistral_filename)
        print(f"  Added {mistral_filename}")

# Check zip file size
zip_size = os.path.getsize(zip_filename) / 1024 / 1024
print(f"\n✅ Created {zip_filename} ({zip_size:.1f} MB)")

# Download individual files
print(f"\nDownloading files...")
files.download(yi_filename)
files.download(mistral_filename)
files.download(zip_filename)

print("\n⬇️ Downloads should start now. Check your Downloads folder!")
print("\n📁 Upload these files to your repository:")
print(f"   data/contrastive_pairs/{yi_filename}")
print(f"   data/contrastive_pairs/{mistral_filename}")